# Operations, Security & Performance Tuning

## What's covered

- Monitoring — JMX, the handful of metrics that actually matter, what "red" looks like
- Consumer lag — the single most important number, and how to compute it from code
- Cluster admin — describing the cluster, adding/removing brokers, partition reassignment
- Rolling upgrades — broker by broker, without dropping traffic
- Security — the four layers (encryption, authentication, authorization, encryption-at-rest)
- TLS — wire encryption between clients and brokers and between brokers
- SASL — `PLAIN`, `SCRAM`, `OAUTHBEARER`, mTLS — when to pick which
- ACLs — the resource/operation matrix, `kafka-acls`, and the AdminClient API
- Quotas — producer, consumer, and connection quotas; per-user, per-client-id, per-IP
- Performance tuning — producer-side, consumer-side, broker-side
- Capacity planning — rough heuristics for partitions, brokers, retention
- The production checklist

## Monitoring — JMX, in one diagram

Every Kafka broker exposes metrics over **JMX (Java Management Extensions)** on a configurable port. The metric surface is enormous — hundreds of MBeans — but the operational reality is that **a small set of metrics tell you 90% of what you need to know.** You get them out of JMX one of three ways:

- **JMX exporter for Prometheus** — the standard. Run the JMX exporter as a Java agent inside each broker JVM; it scrapes MBeans and exposes them at `/metrics`. Prometheus scrapes that, Grafana renders it.
- **Confluent Control Center / Confluent Cloud dashboards** — same data, packaged commercial UI.
- **`jconsole` / `jmxterm`** — interactive, useful for debugging one broker. Not a monitoring strategy.

Same approach for clients: every Kafka producer, consumer, Streams app, and Connect worker exposes JMX metrics too. Scrape them all.

**One alert always:** anyone running Kafka in production should page on **under-replicated partitions > 0 sustained**. That's the cluster telling you a broker is behind or down.

## The metrics that actually matter

Memorize this short list. Everything else is supporting evidence.

**Broker health**

| Metric | What it tells you | What to alert on |
|---|---|---|
| `UnderReplicatedPartitions` | Partitions where ISR < replication factor | `> 0` for more than a few minutes |
| `OfflinePartitionsCount` | Partitions with no leader | `> 0` immediately — these reject writes |
| `ActiveControllerCount` | Per broker; sum across cluster must = 1 | `sum != 1` |
| `RequestHandlerAvgIdlePercent` | Idle time of request-handler thread pool | `< 30%` → broker is CPU-saturated |
| `NetworkProcessorAvgIdlePercent` | Same, for network threads | `< 30%` → network-saturated |
| `BytesInPerSec` / `BytesOutPerSec` | Throughput per topic | Capacity planning, not alerts |
| `MessagesInPerSec` | Records produced per second | Capacity planning |
| `LeaderElectionRateAndTimeMs` | Leader elections / second | Spikes during failures, otherwise zero |
| `UncleanLeaderElectionsPerSec` | Non-ISR replicas elected leader | **`> 0` is data loss** — page immediately |

**Replication health**

| Metric | Meaning |
|---|---|
| `IsrShrinksPerSec` | Followers falling out of the ISR — short bursts are fine, sustained is a problem |
| `IsrExpandsPerSec` | Followers catching back up |
| `FetcherLagMetrics` | Per-partition lag of each follower behind the leader |

**Client side**

| Metric | Meaning |
|---|---|
| `record-send-rate` / `record-error-rate` | Producer throughput and failures |
| `request-latency-avg` / `record-queue-time-avg` | Producer latency; queue-time growth = back-pressure |
| `records-lag-max` / `records-lag` | **Consumer lag** — the single most important client metric |
| `commit-rate` / `commit-latency-avg` | Consumer offset commits |

The honest production dashboard: under-replicated partitions, offline partitions, request-handler idle %, network-handler idle %, ISR shrinks/sec, unclean leader elections, and per-consumer-group lag. That's the boring dashboard that catches the loud failures.

## Consumer lag — the most important number

Consumer lag is the difference between the latest offset in a partition and the consumer group's committed offset for that partition. **It's the answer to "is this consumer keeping up?"**

- **Lag at zero, stable** — consumer is keeping up.
- **Lag stable but nonzero** — consumer is steady but offset behind. Sometimes desirable (batch consumers); for real-time pipelines, undesirable.
- **Lag growing** — consumer is falling behind. Scale up consumers, or upstream is in a burst.

Below: compute lag for every consumer group from Python. The same shape works against any cluster — give it bootstrap servers and read.

In [ ]:
from confluent_kafka import TopicPartition, OFFSET_END, ConsumerGroupTopicPartitions
from confluent_kafka.admin import AdminClient, ConsumerGroupListing

BOOTSTRAP = "localhost:9092"
admin = AdminClient({"bootstrap.servers": BOOTSTRAP})

# 1. List every consumer group on the cluster.
groups = admin.list_consumer_groups().result().valid
group_ids = [g.group_id for g in groups]
print("groups:", group_ids)

for gid in group_ids:
    # 2. Fetch committed offsets for the group.
    offsets_fut = admin.list_consumer_group_offsets([ConsumerGroupTopicPartitions(gid)])
    committed = offsets_fut[gid].result().topic_partitions   # list[TopicPartition]
    if not committed:
        continue

    # 3. For each (topic, partition), fetch the current end offset and compute lag.
    end_req = {tp: OFFSET_END for tp in committed}
    end_fut = admin.list_offsets(end_req)

    print(f"\nGROUP {gid}")
    print(f"  {'topic':<24} {'p':>2}  {'committed':>10}  {'end':>10}  {'lag':>10}")
    for tp, fut in end_fut.items():
        end = fut.result().offset
        committed_off = next(c.offset for c in committed
                             if c.topic == tp.topic and c.partition == tp.partition)
        lag = end - committed_off if committed_off >= 0 else end
        print(f"  {tp.topic:<24} {tp.partition:>2}  {committed_off:>10}  {end:>10}  {lag:>10}")

## Describing the cluster

`AdminClient.describe_cluster()` gives you the controller broker, every broker in the cluster, and the cluster ID — the basis for every operational tool that talks to Kafka.

In [ ]:
desc = admin.describe_cluster().result()

print(f"cluster id : {desc.cluster_id}")
print(f"controller : {desc.controller.id}  ({desc.controller.host}:{desc.controller.port})")
print("\nbrokers:")
for b in desc.nodes:
    print(f"  id={b.id:>3}  endpoint={b.host}:{b.port}  rack={b.rack or '-'}")

## Partition reassignment

Adding or removing brokers doesn't *automatically* rebalance existing partitions — the new broker sits there with no partitions until you move some onto it. **Partition reassignment** is the mechanism:

1. Generate a plan: which partitions to move, to which brokers.
2. Submit the plan; the controller orchestrates moves in the background — each follower replica catches up before the old one is dropped.
3. Verify completion.

The standard CLI tool is `kafka-reassign-partitions.sh`:

```bash
# Generate a balanced plan for these topics across these brokers
kafka-reassign-partitions.sh --bootstrap-server localhost:9092 \
    --topics-to-move-json-file topics.json \
    --broker-list "1,2,3,4" \
    --generate > plan.json

# Execute the plan, optionally throttled to limit replication bandwidth
kafka-reassign-partitions.sh --bootstrap-server localhost:9092 \
    --reassignment-json-file plan.json \
    --execute --throttle 50000000   # 50 MB/s per broker

# Watch progress
kafka-reassign-partitions.sh --bootstrap-server localhost:9092 \
    --reassignment-json-file plan.json --verify
```

**Use the throttle.** Without it, replication can saturate the network and starve client traffic. Tune the throttle to leave headroom for your real workload.

Confluent's **Self-Balancing Clusters** (commercial) and the open-source **Cruise Control** do this automatically — continuously evaluating cluster balance and moving partitions in the background. For a self-managed cluster of any real size, one of those is worth the integration cost.

## Rolling upgrades

Upgrade brokers one at a time. The protocol is designed for it:

1. **Set `inter.broker.protocol.version` and `log.message.format.version`** to your current Kafka version on every broker (if not already pinned). This freezes the wire protocol so newer brokers downgrade to it when talking to old ones.
2. **Stop one broker → upgrade its software → start it.** Replicas it owned go under-replicated; the cluster keeps running.
3. **Wait for `UnderReplicatedPartitions` to return to zero** before moving to the next broker. Don't proceed if replication is still catching up.
4. **Repeat for every broker.**
5. **Bump `inter.broker.protocol.version` to the new version.** This is the cutover — once all brokers are on the new version *and* the protocol is bumped, the new protocol features become active. Roll restart one more time to apply.
6. **(Optional, separate cycle) Bump `log.message.format.version`** once all producers are on the new version too.

The pattern: software upgrade and protocol cutover are separate steps, and you wait for replication to be healthy between every broker.

## Security — the four layers

Production Kafka security is four independent concerns. Pick a position on each.

| Layer | What it protects | Mechanism |
|---|---|---|
| **Encryption in transit** | Anyone tapping the network | TLS (SSL) on broker listeners |
| **Authentication** | "Who is this client?" | SASL (PLAIN, SCRAM, OAUTHBEARER) or mTLS |
| **Authorization** | "Can this client do this thing?" | ACLs |
| **Encryption at rest** | Attacker with disk access | Disk-level encryption (LUKS, EBS encryption); Kafka does not encrypt its own log files |

**You can mix and match.** Common production posture: TLS everywhere, SASL/SCRAM for app clients, mTLS for broker-to-broker, ACLs deny-by-default. Cloud managed offerings (MSK, Confluent Cloud) handle most of this configuration for you; self-managed clusters need it set up explicitly.

## TLS — wire encryption

TLS terminates on the broker listener. Configured per listener via the `security.protocol`:

- **`PLAINTEXT`** — no TLS, no auth. Local dev only.
- **`SSL`** — TLS encryption; optional client cert (mTLS) for authentication.
- **`SASL_PLAINTEXT`** — SASL auth, no encryption. Internal networks only.
- **`SASL_SSL`** — SASL auth over TLS. The standard production choice.

Broker config (`server.properties`):

```properties
listeners=INTERNAL://0.0.0.0:9091,EXTERNAL://0.0.0.0:9092
listener.security.protocol.map=INTERNAL:SSL,EXTERNAL:SASL_SSL
inter.broker.listener.name=INTERNAL

ssl.keystore.location=/etc/kafka/secrets/broker.keystore.jks
ssl.keystore.password=...
ssl.truststore.location=/etc/kafka/secrets/broker.truststore.jks
ssl.truststore.password=...
ssl.client.auth=required    # set to 'required' if using mTLS for client auth
```

Client config (Python):

```python
Producer({
    "bootstrap.servers": "broker.example.com:9092",
    "security.protocol": "SASL_SSL",
    "ssl.ca.location": "/etc/ssl/ca.pem",
    "sasl.mechanism": "SCRAM-SHA-512",
    "sasl.username": "payments-producer",
    "sasl.password": os.environ["KAFKA_PASSWORD"],
})
```

**Multiple listeners are normal.** Brokers often expose an internal listener (PLAINTEXT or SSL, for cluster-internal traffic) and an external listener (SASL_SSL, for clients). Configure `advertised.listeners` so clients connect on the right one.

## SASL — who is this client?

Four SASL mechanisms commonly used with Kafka:

| Mechanism | Credentials | When to pick |
|---|---|---|
| **`PLAIN`** | Username + password in plaintext (encrypted by TLS) | Only over TLS, with a static credential store; the simplest option |
| **`SCRAM-SHA-256` / `SCRAM-SHA-512`** | Username + password, salted and hashed | The recommended username/password mechanism. Credentials stored in `__consumer_offsets`-style internal Kafka or external store |
| **`OAUTHBEARER`** | OIDC token from an identity provider (Okta, Auth0, Azure AD) | Service-to-service auth in OIDC environments; ties Kafka access to your IdP |
| **mTLS** | Client certificate | Strong identity bound to certificate issuance; common in Kubernetes service-mesh deployments |

**The default recommendation for a self-managed cluster:** SCRAM-SHA-512 over TLS, with credentials provisioned by your secrets system. For OIDC-native organizations, OAUTHBEARER is cleaner. mTLS shines when certificate management is already a solved problem (e.g. SPIFFE/SPIRE).

**Once a client authenticates, Kafka maps it to a *principal*** — typically `User:CN=payments-producer` for mTLS or `User:payments-producer` for SASL. Principals are what ACLs are written against.

## ACLs — can this client do this thing?

An **ACL** is a tuple: `(principal, resource, operation, permission)`. Kafka resources are typed:

| Resource type | Operations that apply |
|---|---|
| `Topic` | `Read`, `Write`, `Create`, `Delete`, `Alter`, `Describe`, `DescribeConfigs`, `AlterConfigs` |
| `Group` (consumer group) | `Read`, `Describe`, `Delete` |
| `Cluster` | `Create` (topics), `Alter`, `Describe`, `ClusterAction`, `IdempotentWrite` |
| `TransactionalId` | `Write`, `Describe` |
| `DelegationToken` | `Describe` |

**Deny-by-default.** Set `allow.everyone.if.no.acl.found=false` on the broker, then write explicit allow rules per principal. Anything not in the matrix is denied.

The CLI:

```bash
kafka-acls.sh --bootstrap-server localhost:9092 \
    --add --allow-principal User:payments-producer \
    --operation Write --topic payments.created

kafka-acls.sh --bootstrap-server localhost:9092 \
    --add --allow-principal User:payments-aggregator \
    --operation Read --topic payments.created \
    --operation Read --group payments-aggregator-v1
```

Same operations exist on `AdminClient.create_acls(...)` / `describe_acls(...)` / `delete_acls(...)` — useful for managing ACLs as code from CI.

## Quotas — one tenant can't take down the cluster

Quotas limit per-client throughput. Without them, one runaway producer (or one runaway test) can saturate broker network or disk and degrade everyone.

Three quota dimensions, set per `user`, `client-id`, IP, or combinations:

- **Produce rate** — bytes/second a producer can write. Exceed it and the broker delays responses (back-pressure).
- **Fetch rate** — bytes/second a consumer can read.
- **Request rate** — CPU time the broker spends on this client (percentage of one network or I/O thread).
- **Connection rate** — new connections per second from one IP. Defends against connection floods.

Set via the CLI:

```bash
kafka-configs.sh --bootstrap-server localhost:9092 \
    --alter --entity-type users --entity-name payments-producer \
    --add-config 'producer_byte_rate=10485760,consumer_byte_rate=10485760'
```

Quotas don't reject requests — they slow them down. The client sees increased request latency; the broker stays healthy. **In a multi-tenant cluster, set per-user default quotas at provisioning time.**

## Performance tuning — producers

(Most of this was covered in notebook 02; here it is in tuning-knob form.)

**For throughput:** raise `linger.ms` (default 0; try 5–20), raise `batch.size` (default 16 KB; try 64–256 KB), set `compression.type=zstd`, raise `buffer.memory` if you see queue-time growing.

**For latency:** keep `linger.ms=0` (the default), prefer fewer in-flight requests, keep records small.

**For durability:** `acks=all`, `enable.idempotence=true` (both default since 3.0), broker-side `min.insync.replicas=2`. Combined with `replication.factor=3` this is the standard production recipe.

**Almost never tune:** `retries`, `request.timeout.ms`. Trust `delivery.timeout.ms` as the single bound on retry behavior.

## Performance tuning — consumers

**For throughput:** raise `fetch.min.bytes` (default 1; try 64 KB) and `fetch.max.wait.ms` to batch more records per fetch. Raise `max.poll.records` (default 500) if your handler is fast enough.

**For latency:** keep `fetch.min.bytes` at 1, keep `fetch.max.wait.ms` low.

**For correctness:** `enable.auto.commit=false` and manual commit after processing — the at-least-once recipe from notebook 03.

**For rebalances:** `partition.assignment.strategy=cooperative-sticky` and `group.instance.id` set to the pod/host name. Rolling restarts stop causing rebalance storms.

**The most common consumer problem isn't broker-side; it's the handler being slow.** Watch `max.poll.interval.ms` — anything close to its bound (default 5 minutes) means your handler is the bottleneck.

## Performance tuning — brokers

Where the real money is. A short list of the knobs that matter:

- **OS page cache is your storage.** Kafka relies on the kernel page cache for read speed. Give the broker box generous RAM and *don't* set a huge JVM heap to consume it. **JVM heap on broker: 6–8 GB is plenty.** The rest of physical RAM should belong to the page cache.
- **`num.network.threads`** (default 3) — handle network I/O. Scale roughly with client connection count.
- **`num.io.threads`** (default 8) — handle disk reads/writes. Scale roughly with disk count and partition activity.
- **`num.replica.fetchers`** (default 1) — followers fetching from leaders. Raise to 4–8 for clusters with many partitions or wide replication.
- **`log.flush.*` — almost never set.** Kafka deliberately relies on the kernel to flush dirty pages on its own schedule. Forcing synchronous flushes hurts throughput dramatically with no durability gain (replication, not fsync, is your durability story).
- **Disks: prefer one disk per `log.dirs` entry.** Kafka spreads partitions across log directories; one drive each lets you scale by adding drives, with no RAID overhead. SSDs are nice; for sequential workloads, well-tuned spinning rust is also fine.
- **JVM: G1GC is the default and works well. ZGC for very large heaps**, but most brokers don't need that.
- **`socket.send.buffer.bytes` / `socket.receive.buffer.bytes`** (default 100 KB) — bump on high-latency / high-throughput links.

## Capacity planning — rough heuristics

Back-of-envelope numbers to start from. Validate on your hardware.

**Per broker, commodity 16-core/64 GB box with NVMe SSDs:**

- **Sustained throughput**: 100–500 MB/s ingress, 200 MB/s–1 GB/s egress (consumers replay-from-cold-storage skew this).
- **Partition count**: 2,000–4,000 partitions per broker before the metadata burden hurts. KRaft pushes this much higher (tens of thousands), but file-handle and replication overhead still grow linearly.
- **Connection count**: thousands.

**Cluster sizing math** for a target throughput `T` MB/s with replication factor 3:

```text
  broker_ingress_required = T (replication multiplies inside the cluster but ingress is T)
  broker_egress_required  = T * (consumer_groups + (replication_factor - 1))
  brokers_min             = max(broker_ingress_required, broker_egress_required) / per_broker_throughput
```

Add 30–50% headroom for failures: you need spare capacity to absorb the lost broker's traffic during a node failure.

**Disk sizing.** Multiply target throughput by retention to get raw disk per broker; multiply by replication factor for cluster-wide; divide by broker count for per-broker. Compression cuts this by 2–10x. Tiered storage (notebook 04) decouples disk sizing from retention entirely.

## The production checklist

Print this and pin it next to the deployment runbook.

**Cluster**

- [ ] At least 3 brokers, KRaft mode (or 3-node ZooKeeper if you're still on it)
- [ ] `broker.rack` set per broker (the AZ/zone)
- [ ] Multiple listeners — internal (SSL or PLAINTEXT) + external (SASL_SSL)
- [ ] `unclean.leader.election.enable=false`
- [ ] `auto.create.topics.enable=false`
- [ ] JMX metrics scraped into Prometheus; dashboard for under-replicated, offline, unclean elections, request idle %, consumer lag
- [ ] Alerts on under-replicated > 0 sustained, offline > 0, unclean elections > 0, controller count != 1

**Topics**

- [ ] `replication.factor=3`
- [ ] `min.insync.replicas=2`
- [ ] Partition count sized for peak consumer parallelism + headroom
- [ ] `cleanup.policy` chosen deliberately (delete vs compact)
- [ ] Retention set (time and/or size)
- [ ] Naming convention enforced (`<domain>.<entity>.<event>.<version>`)

**Producers**

- [ ] `acks=all`, `enable.idempotence=true` (defaults since 3.0)
- [ ] `compression.type=zstd`, `linger.ms ≥ 5`
- [ ] Delivery callbacks; `flush()` on shutdown
- [ ] If exactly-once: `transactional.id` set, stable across restarts

**Consumers**

- [ ] `enable.auto.commit=false`; commit after processing
- [ ] `partition.assignment.strategy=cooperative-sticky`
- [ ] `group.instance.id` set for static membership
- [ ] Per-group lag dashboard + alert

**Schemas**

- [ ] Schema Registry HA; `_schemas` topic on RF=3
- [ ] Compatibility mode set per-subject (default `BACKWARD`)
- [ ] One Schema Registry per environment

**Security**

- [ ] TLS on external listener (and ideally internal too)
- [ ] SASL mechanism chosen (SCRAM-SHA-512 / OAUTHBEARER / mTLS)
- [ ] `allow.everyone.if.no.acl.found=false` — deny by default
- [ ] Per-principal ACLs as code
- [ ] Default quotas at provisioning time

**Operations**

- [ ] Backup story (MirrorMaker 2 to a DR cluster, or tiered storage)
- [ ] Rolling-upgrade runbook tested
- [ ] Partition reassignment script + throttle defaults
- [ ] Cruise Control or equivalent for ongoing balance
- [ ] Incident playbook for under-replicated, offline, controller flapping

## Common gotchas

- **`unclean.leader.election.enable=true` to "avoid downtime."** Silent data loss. Already covered in notebook 04 — worth saying again.
- **Huge JVM heap on brokers.** Robs the page cache. 6–8 GB is plenty; let physical RAM serve disk reads.
- **Auto-create topics on in production.** Typos in client code create garbage topics. Disable it.
- **No quotas in multi-tenant clusters.** One bad tenant degrades all others. Set defaults at provisioning.
- **Single-AZ deployment.** Survives broker failure but not AZ failure. Three AZs with `broker.rack` set is the standard.
- **No monitoring on Schema Registry or Connect.** Both can fail independently of brokers. They need their own dashboards and alerts.
- **Partition reassignment without a throttle.** Replication saturates the network and starves client traffic.
- **Forgetting that `min.insync.replicas` is broker-level and per-topic.** Setting it broker-wide doesn't override a topic that was created with a lower value. Set both.
- **Trusting `acks=all` without `min.insync.replicas`.** When the ISR shrinks to 1, `acks=all` collapses to `acks=1` silently. Always pair them.

## Closing the loop

The curriculum is complete. Eight notebooks, end to end:

1. **Foundations & Architecture** — brokers, partitions, offsets, replication, KRaft
2. **Producers** — keys, partitioner, `acks`, idempotence, transactions, batching, compression
3. **Consumers & Consumer Groups** — poll loop, rebalance, commits, isolation, static membership
4. **Topics, Partitions & Storage** — sizing, retention, compaction, segments, the durability recipe
5. **Schema Registry & Serialization** — Avro/Protobuf/JSON Schema, the wire format, compatibility modes
6. **Kafka Connect** — source/sink connectors, REST, SMTs, DLQs
7. **Kafka Streams** — KStream/KTable, topology, state stores, joins, windowing, EOS v2
8. **Operations, Security & Performance Tuning** — JMX, lag, ACLs, TLS, SASL, the production checklist

Together they cover the surface the Confluent Certified Developer for Apache Kafka exam tests and — more usefully — the surface a real team needs to run Kafka in production. From here, the next steps are usually one of:

- **Deepen one area** — Debezium and CDC patterns, Kafka Streams' Processor API, ksqlDB, or the broker internals (the controller, the log layer, KRaft's metadata topic).
- **Build something.** Pick a small domain — a payment pipeline, a clickstream aggregator, an order fulfillment workflow — and wire it end to end with what you've learned.

The certification studies for itself once the eight notebooks land. The actual skill is the muscle memory of designing topics, picking serdes, and keeping the production cluster boring.